In [ ]:
import os
import sys
from datetime import datetime
import pandas as pd
import polars as pl

In [ ]:
try: 
    df = pd.read_csv('pokemon_stats_2025.csv')
except FileNotFoundError:
    print("Error: 'pokemon_stats_2025.csv' file not found.")

## Data Exploration: 

In [ ]:
print("Total number of rows and columns:")
print(f'{df.shape}\n')

print(f'Column names: {df.columns.tolist()}\n')

print("Data types of each column:")
print(df.dtypes)

print('\nEmpty cells in each column:')
print(df.isnull().sum())

#### Conclusion: 
"type_2" has empty value and that is to be expected since not all pokemons have a second type.

### Duplicat Values

In [ ]:
print(df.describe(include='all'))

print('\nOverall unique values in each column:')
print(df.nunique())

In [ ]:
#  To do next: 
# create a new df that will hold the same information as original. It will also hold a total stats column. 
# A new df that is only for Pokemon that have one type. Another one that has two types.

* Following a tutorial on Filtering and Ordering: https://www.youtube.com/watch?v=kB7FV-ijdqE
* Following the Index Tutorial: https://www.youtube.com/watch?v=mBCG9J1TVTc



## Pokemon Stats Analysis

In [ ]:
stat_cols = ['hp', 'attack', 'defense', 'special_attack', 'special_defense', 'speed']
df_stats = df.copy()
df_stats['total_stats'] = df_stats[stat_cols].sum(axis=1)

print("=== Top 10 Pokemon by Total Stats ===")
top10 = df_stats[['name', 'type_1', 'type_2', 'total_stats'] + stat_cols].sort_values('total_stats', ascending=False).head(10)
print(top10.to_string(index=False))


In [ ]:
print("=== Pokemon with Highest Stat in Each Category ===")
for stat in stat_cols + ['total_stats']:
    row = df_stats.loc[df_stats[stat].idxmax()]
    print(f"{stat:>15}: {row['name']} ({row[stat]})")


In [ ]:
print("=== Average Stats by Primary Type ===")
avg_by_type = df_stats.groupby('type_1')[stat_cols + ['total_stats']].mean().round(1).sort_values('total_stats', ascending=False)
print(avg_by_type.to_string())


In [ ]:
print("=== Single-Type vs Dual-Type Pokemon: Average Stats ===")
df_single = df_stats[df_stats['type_2'].isna()]
df_dual   = df_stats[df_stats['type_2'].notna()]

comparison = pd.DataFrame({
    'single_type': df_single[stat_cols + ['total_stats']].mean().round(1),
    'dual_type':   df_dual[stat_cols + ['total_stats']].mean().round(1),
})
comparison['difference'] = (comparison['dual_type'] - comparison['single_type']).round(1)
print(f"Single-type count: {len(df_single)}  |  Dual-type count: {len(df_dual)}\n")
print(comparison.to_string())


## ML: Predicting `type_1` from Battle Stats (Single-Type Pokemon)

In [ ]:
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, f1_score
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, f1_score
from sklearn.pipeline import Pipeline
warnings.filterwarnings('ignore')

In [ ]:
stat_cols = ['hp', 'attack', 'defense', 'special_attack', 'special_defense', 'speed']

# Filter to single-type Pokemon only
df_single = df[df['type_2'].isna()].copy()

# Feature engineering
df_single['total_stats']     = df_single[stat_cols].sum(axis=1)
df_single['atk_spatk_ratio'] = df_single['attack'] / (df_single['special_attack'] + 1)
df_single['bulk_index']      = (df_single['hp'] + df_single['defense'] + df_single['special_defense']) / 3

# Drop ultra-rare classes (n < 5) to allow stratified splitting
class_counts = df_single['type_1'].value_counts()
valid_classes = class_counts[class_counts >= 5].index
df_model = df_single[df_single['type_1'].isin(valid_classes)].copy()

print(f"Single-type Pokemon: {len(df_single)}")
print(f"After dropping rare classes (<5 samples): {len(df_model)}")
print(f"\nClasses retained ({len(valid_classes)}):")
print(df_model['type_1'].value_counts())

In [ ]:
feature_cols = stat_cols + ['total_stats', 'atk_spatk_ratio', 'bulk_index']
X = df_model[feature_cols]
y = df_model['type_1']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

### Model Comparison via Cross-Validation

In [ ]:
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(class_weight='balanced', C=1.0,
                                   max_iter=1000, random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(n_estimators=300, max_depth=7,
                                        class_weight='balanced', random_state=42))
    ]),
    'HistGradientBoosting': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05,
                                                max_iter=200, random_state=42))
    ]),
    'KNN': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', KNeighborsClassifier(n_neighbors=5, weights='distance'))
    ]),
}

print(f"{'Model':30s}  Macro-F1 (5-fold CV)")
print("-" * 50)
cv_results = {}
for name, pipeline in models.items():
    scores = cross_val_score(pipeline, X_train, y_train,
                             cv=cv, scoring='f1_macro', n_jobs=-1)
    cv_results[name] = scores
    print(f"{name:30s}  {scores.mean():.3f} ± {scores.std():.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot([cv_results[m] for m in models], labels=list(models.keys()))
ax.set_ylabel('Macro F1 (5-fold CV)')
ax.set_title('Model Comparison — Single-Type Pokemon Type Prediction')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

### Best Model — Final Evaluation on Hold-Out Test Set

In [ ]:
# Pick the model with the highest mean CV score
best_name = max(cv_results, key=lambda m: cv_results[m].mean())
print(f"Best model: {best_name}")

best_model = models[best_name]
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

print(f"\nMacro F1 on test set: {f1_score(y_test, y_pred, average='macro'):.3f}")
print(f"Weighted F1 on test set: {f1_score(y_test, y_pred, average='weighted'):.3f}")
print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, ax=ax, normalize='true',
    colorbar=False, xticks_rotation=45
)
ax.set_title(f'Normalized Confusion Matrix — {best_name}')
plt.tight_layout()
plt.show()

### Feature Importance

In [ ]:
# Feature importance works for tree-based models; fall back to coefficients for linear
clf_step = best_model.named_steps['clf']

if hasattr(clf_step, 'feature_importances_'):
    importances = pd.Series(clf_step.feature_importances_, index=feature_cols)
    title = f'Feature Importances — {best_name}'
elif hasattr(clf_step, 'coef_'):
    importances = pd.Series(clf_step.coef_.mean(axis=0), index=feature_cols)
    title = f'Mean Coefficient Magnitude — {best_name}'
else:
    importances = None

if importances is not None:
    importances.sort_values().plot(kind='barh', figsize=(7, 4), color='steelblue')
    plt.xlabel('Importance')
    plt.title(title)
    plt.tight_layout()
    plt.show()